# Buoc 1: Fix BestBuy Filter (Replace requests with Playwright)

**Goal:** `requests.get()` bi BestBuy block -> timeout 10/10.  
**Solution:** Dung Playwright, kiem tra sale + cao du lieu ngay trong 1 lan truy cap.  
**Return:** `List[Tuple[str, dict]]` giong Amazon.

## Cell 1: Reproduce bug - requests.get() bi block

In [ ]:
import requests
import time

test_urls = [
    "https://www.bestbuy.com/product/asus-zenbook-a14-14-fhd-oled-laptop-copilot-pc-snapdragon-x-plus-16gb-ram-512gb-ssd-zabriskie-beige/JJGGLH86J4",
    "https://www.bestbuy.com/product/asus-zenbook-14-14-fhd-oled-touch-screen-laptop-intel-core-ultra-7-16gb-ram-512gb-ssd-jasper-gray/JJGGLH7HXW",
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

for url in test_urls:
    start = time.time()
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        elapsed = time.time() - start
        print(f"[{elapsed:.1f}s] Status: {resp.status_code} | Length: {len(resp.content)} | URL: {url[:70]}")
    except Exception as e:
        elapsed = time.time() - start
        print(f"[{elapsed:.1f}s] ERROR: {e} | URL: {url[:70]}")

## Cell 2a: Test thu browser nao load duoc bestbuy.com

Chromium bi `ERR_HTTP2_PROTOCOL_ERROR`. Thu Firefox va Chrome channel.

In [ ]:
import time
from playwright.async_api import async_playwright

TEST_URL = "https://www.bestbuy.com/product/asus-zenbook-a14-14-fhd-oled-laptop-copilot-pc-snapdragon-x-plus-16gb-ram-512gb-ssd-zabriskie-beige/JJGGLH86J4"

async def try_browser(browser_type_name: str, **launch_kwargs):
    """Try loading BestBuy with a specific browser."""
    async with async_playwright() as p:
        browser_type = getattr(p, browser_type_name)
        print(f"\n{'='*60}")
        print(f"Testing: {browser_type_name} {launch_kwargs}")
        print(f"{'='*60}")

        try:
            browser = await browser_type.launch(headless=False, **launch_kwargs)
            context = await browser.new_context(
                user_agent=(
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:134.0) "
                    "Gecko/20100101 Firefox/134.0"
                ) if browser_type_name == "firefox" else (
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/131.0.0.0 Safari/537.36"
                ),
                viewport={"width": 1920, "height": 1080},
                locale="en-US",
            )
            page = await context.new_page()

            # Try homepage first
            start = time.time()
            print("1) Loading homepage...")
            await page.goto("https://www.bestbuy.com", timeout=30000, wait_until="domcontentloaded")
            print(f"   Homepage OK ({time.time()-start:.1f}s) | Title: {await page.title()}")

            # Try product page
            start = time.time()
            print("2) Loading product page...")
            await page.goto(TEST_URL, timeout=30000, wait_until="domcontentloaded")
            await page.wait_for_timeout(2000)
            title = await page.title()
            print(f"   Product OK ({time.time()-start:.1f}s) | Title: {title}")

            # Check if we got real content
            h1 = page.locator("h1")
            if await h1.count() > 0:
                h1_text = await h1.first.text_content()
                print(f"   H1: {h1_text[:80]}")

            await browser.close()
            return True

        except Exception as e:
            print(f"   FAILED: {e}")
            try:
                await browser.close()
            except:
                pass
            return False

# Test 1: Firefox
await try_browser("firefox")

# Test 2: Chromium with --disable-http2
await try_browser("chromium", args=["--disable-http2", "--no-sandbox", "--disable-blink-features=AutomationControlled"])

# Test 3: Chrome channel (real Chrome, if installed)
try:
    await try_browser("chromium", channel="chrome", args=["--no-sandbox"])
except Exception as e:
    print(f"\nChrome channel not available: {e}")

## Cell 3: Test voi 2 URLs

In [ ]:
import time

test_urls = [
    "https://www.bestbuy.com/product/asus-zenbook-a14-14-fhd-oled-laptop-copilot-pc-snapdragon-x-plus-16gb-ram-512gb-ssd-zabriskie-beige/JJGGLH86J4",
    "https://www.bestbuy.com/product/asus-zenbook-14-14-fhd-oled-touch-screen-laptop-intel-core-ultra-7-16gb-ram-512gb-ssd-jasper-gray/JJGGLH7HXW",
]

start = time.time()
results = await filter_and_scrape_bestbuy_playwright(test_urls, headless=False)
elapsed = time.time() - start

print(f"\n--- Total time: {elapsed:.1f}s ---")
print(f"--- Sale items: {len(results)}/{len(test_urls)} ---")

## Cell 4: Xem chi tiet ket qua

In [ ]:
for url, info in results:
    print("=" * 80)
    print(f"URL:      {url}")
    print(f"Title:    {info.get('title', 'N/A')}")
    print(f"Brand:    {info.get('brand', 'N/A')}")
    print(f"Sale $:   ${info.get('sale_price', 0):.2f}")
    print(f"Was $:    {info.get('original_price', 'N/A')}")
    print(f"Savings:  {info.get('savings', 'N/A')}")
    print(f"Features: {info.get('features', '')[:200]}...")
    print()